# NBA en diferido · sin spoilers
Ejecuta las celdas en orden. Si faltan dependencias, ejecuta `%pip install requests tzdata` en una celda antes de empezar.

Solo se muestran equipo, rival y recomendación: **ver completo**, **resumen** o **prescindible**. No se muestran resultados, notas, ganadores, prórrogas ni explicaciones del desenlace. Orden alfabético.

Consulta ESPN por fechas y evalúa únicamente partidos finalizados cuyo inicio esté entre ayer a las 06:30 y hoy a las 06:30, hora de Madrid. Antes del cierre, la cobertura puede ser parcial. Puedes volver a ejecutarlo más tarde.

Conserva tus afinidades y los umbrales originales. La clasificación tiene un valor neutral de 0,5 hasta disponer de una fuente validada: no se inventan puestos ni se mezclan conferencias. El margen final pesa un 65%, clasificación neutral 20%, afinidad 10% y puntos totales 5%. Los puntos totales son una aproximación heredada, no una medida de posesiones. Estas etiquetas son orientativas; no equivalen a las notas de la agenda en directo.

No hay credenciales ni envío de correo desde este cuaderno. La consulta depende de un endpoint público de ESPN que podría cambiar.


In [ ]:
"""NBA en diferido: la salida pública no contiene marcadores ni notas."""
import json
import time
from datetime import datetime, timedelta, time as hora, timezone
from pathlib import Path

import requests
from zoneinfo import ZoneInfo
MADRID_TZ = ZoneInfo("Europe/Madrid")

URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard'
AFINIDADES = {'Celtics': 0.85, 'Warriors': 0.75, 'Nuggets': 0.7, 'Timberwolves': 0.7, 'Spurs': 0.6, 'Rockets': 0.6, 'Pacers': 0.55, 'Raptors': 0.5, 'Hornets': 0.45, 'Pelicans': 0.35, 'Clippers': 0.3, 'Kings': 0.3, 'Bucks': 0.2, 'Hawks': 0.15, 'Nets': 0.15, 'Bulls': 0.1, 'Magic': 0.05, 'Wizards': 0.0, 'Pistons': 0.0, 'Jazz': 0.0, 'Trail Blazers': 0.0, 'Thunder': -0.45, 'Knicks': -0.4, 'Heat': -0.4, '76ers': -0.35}


class ErrorNBA(Exception):
    pass


def descargar(dia):
    for intento in range(3):
        try:
            r = requests.get(URL, params={'dates': dia.strftime('%Y%m%d'), 'limit': 100}, timeout=(5, 20))
            if r.status_code == 429 or r.status_code >= 500:
                if intento < 2:
                    time.sleep(2 ** intento)
                    continue
            r.raise_for_status()
            datos = r.json()
            if not isinstance(datos, dict) or not isinstance(datos.get('events'), list):
                raise ErrorNBA('La fuente NBA ha cambiado de formato.')
            return datos['events']
        except (requests.Timeout, requests.ConnectionError):
            if intento < 2:
                time.sleep(2 ** intento)
                continue
            raise ErrorNBA('No se pudo consultar la NBA tras tres intentos.') from None
        except (requests.RequestException, ValueError):
            raise ErrorNBA('No se pudo leer la fuente NBA.') from None


def puntuacion(local, visitante, puntos_local, puntos_visitante):
    diferencia = abs(puntos_local - puntos_visitante)
    margen = (1 if diferencia <= 2 else .85 if diferencia <= 5 else .75 if diferencia <= 8
              else .5 if diferencia <= 12 else .15)
    gustos = max(0, min(1, .5 + (AFINIDADES.get(local, 0) + AFINIDADES.get(visitante, 0)) / 4))
    total = puntos_local + puntos_visitante
    ritmo = 1 if total >= 230 else .7 if total >= 215 else .4 if total >= 200 else .2
    # Clasificación ausente: contribución neutral, nunca puestos ficticios.
    return .65 * margen + .20 * .5 + .10 * gustos + .05 * ritmo


def evaluar_partidos(partidos, ahora):
    if ahora.tzinfo is None:
        raise ValueError('La fecha NBA debe incluir zona horaria.')
    ahora = ahora.astimezone(MADRID_TZ)
    cierre = datetime.combine(ahora.date(), hora(6, 30), MADRID_TZ)
    inicio = datetime.combine(ahora.date() - timedelta(days=1), hora(6, 30), MADRID_TZ)
    limite = min(ahora, cierre)
    vistos = set()
    salida, pendientes, invalidos = [], 0, 0
    for e in partidos:
        try:
            fecha = datetime.fromisoformat(e['date'].replace('Z', '+00:00'))
            if fecha.tzinfo is None:
                raise ValueError()
            if not inicio <= fecha < cierre or fecha > limite:
                continue
            identificador = str(e['id'])
            if identificador in vistos:
                continue
            vistos.add(identificador)
            if not e['status']['type']['completed']:
                pendientes += 1
                continue
            equipos = e['competitions'][0]['competitors']
            local = next(t for t in equipos if t['homeAway'] == 'home')
            visitante = next(t for t in equipos if t['homeAway'] == 'away')
            puntos = [int(t['score']) for t in (local, visitante)]
            if any(p < 0 for p in puntos) or puntos[0] == puntos[1]:
                raise ValueError()
            nombres = [t['team']['displayName'] for t in (local, visitante)]
            afinidades = [t['team']['name'] for t in (local, visitante)]
            nota = puntuacion(*afinidades, *puntos)
            recomendacion = 'Ver completo' if nota >= .80 else 'Resumen' if nota >= .65 else 'Prescindible'
            # Nunca devolver el objeto de la fuente: contiene spoilers incluso en sus enlaces.
            salida.append({'local': nombres[0], 'visitante': nombres[1],
                           'fecha': fecha.astimezone(MADRID_TZ).strftime('%d/%m %H:%M'),
                           'recomendacion': recomendacion})
        except (KeyError, IndexError, TypeError, ValueError, StopIteration, AttributeError):
            invalidos += 1
    # Orden estable por nombres, nunca por margen/nota oculta.
    salida.sort(key=lambda p: (p['local'], p['visitante']))
    avisos = ['Clasificación no incorporada: ese factor se mantiene neutral.']
    if pendientes:
        avisos.append(f'{pendientes} encuentros iniciados aún sin resultado final confirmado; no se evalúan.')
    if invalidos:
        avisos.append(f'{invalidos} registros no se pudieron validar; cobertura incompleta.')
    if ahora < cierre:
        avisos.append('Consulta antes de las 06:30: la jornada puede seguir en curso y el correo no se actualiza automáticamente.')
    return {'partidos': salida, 'avisos': avisos, 'fuente': 'ESPN',
            'ventana': f'{inicio:%d/%m %H:%M}–{cierre:%d/%m %H:%M} · Madrid',
            'consultado': ahora.strftime('%d/%m %H:%M')}


def generar_nba(ahora=None):
    ahora = ahora or datetime.now(MADRID_TZ)
    if ahora.tzinfo is None:
        raise ValueError('La fecha NBA debe incluir zona horaria.')
    local = ahora.astimezone(MADRID_TZ)
    inicio = datetime.combine(local.date() - timedelta(days=1), hora(6, 30), MADRID_TZ)
    cierre = min(local, datetime.combine(local.date(), hora(6, 30), MADRID_TZ))
    dia = inicio.astimezone(timezone.utc).date()
    ultimo = cierre.astimezone(timezone.utc).date()
    partidos = []
    while dia <= ultimo:
        partidos.extend(descargar(dia))
        dia += timedelta(days=1)
    return evaluar_partidos(partidos, ahora)


In [ ]:
from IPython.display import HTML, display
from html import escape
nba = generar_nba()
filas = ''.join('<tr><td style="padding:12px;border-bottom:1px solid #ddd">' + escape(p['local'] + ' — ' + p['visitante']) + '</td><td style="padding:12px;border-bottom:1px solid #ddd">' + escape(p['recomendacion']) + '</td></tr>' for p in nba['partidos'])
display(HTML('<h2>NBA en diferido · sin spoilers</h2><p>' + escape(nba['ventana']) + '</p><table>' + filas + '</table>'))
if not nba['partidos']:
    print('No hay partidos finalizados y validados en esta consulta.')
for aviso in nba['avisos']:
    print(aviso)
